# TimesFM forecasting for national grid demand

TimesFM is a pretrained decoder-only time-series foundation model. This notebook evaluates Google Research's TimesFM 2.5 200M checkpoint without training it from scratch. A foundation model is useful here because it can produce a zero-shot forecast from the demand history and provides a direct comparison with the project's trained LSTM models.

## 1. Imports and experiment configuration

The experiment uses an hourly series, the previous 168 hours as context, and the next 24 hours as the forecast target.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.timesfm.timesfm_model import generate_forecast, load_timesfm_model
from models.timesfm.timesfm_utils import (
    build_forecast_windows, calculate_metrics, find_hourly_gaps,
    load_demand_data, prepare_timesfm_input, save_evaluation,
    save_model_comparison, save_plots, save_predictions,
)

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'master_training_data.csv'
RESULTS_DIR = PROJECT_ROOT / 'results'
CONTEXT_LENGTH = 168
HORIZON = 24

## 2. Load and prepare the hourly demand series

In [ ]:
demand = load_demand_data(DATA_PATH)
timesfm_input = prepare_timesfm_input(demand)
gaps = find_hourly_gaps(demand)
print(f'{len(demand):,} usable rows; {len(gaps)} timestamp discontinuities')
display(timesfm_input.head())

## 3. Build chronological evaluation windows

Windows begin in the final 15% of the series. Any window containing a non-hourly timestamp step is skipped. The daily stride avoids overlapping target periods; set `max_windows=None` and `stride=1` for every eligible hourly origin.

In [ ]:
contexts, actuals, forecast_times = build_forecast_windows(
    demand, context_length=CONTEXT_LENGTH, horizon=HORIZON,
    test_ratio=0.15, stride=24, max_windows=None,
)
print(f'Evaluation windows: {len(contexts):,}')
print('Context shape:', contexts[0].shape, 'Target shape:', actuals[0].shape)

## 4. Load the pretrained model and forecast

The first execution downloads `google/timesfm-2.5-200m-pytorch`. TimesFM normalizes each input series internally and returns a 24-value point forecast for every 168-value context.

In [ ]:
model = load_timesfm_model(context_length=CONTEXT_LENGTH, horizon=HORIZON, batch_size=32)
predicted = generate_forecast(model, contexts, horizon=HORIZON, batch_size=32)
actual = np.stack(actuals)
predicted.shape

## 5. Evaluate and save results

In [ ]:
metrics = calculate_metrics(actual, predicted)
prediction_data = save_predictions(
    forecast_times, actual, predicted, RESULTS_DIR / 'timesfm_predictions.csv'
)
evaluation = save_evaluation(metrics, RESULTS_DIR / 'timesfm_evaluation_results.csv')
comparison = save_model_comparison(
    PROJECT_ROOT, metrics, RESULTS_DIR / 'model_comparison_timesfm.csv'
)
save_plots(prediction_data, RESULTS_DIR / 'plots' / 'timesfm', HORIZON)
display(evaluation)

## 6. Visualize an example forecast

In [ ]:
example = prediction_data.iloc[:HORIZON]
ax = example.plot(x='timestamp', y=['actual_demand', 'predicted_demand'], marker='o', figsize=(12, 5))
ax.set_title('Example 24-hour TimesFM forecast')
ax.set_ylabel('Demand (MW)')
plt.tight_layout()

## 7. Compare TimesFM with LSTM results

In [ ]:
display(comparison.sort_values('RMSE', na_position='last'))

## 8. Discussion and conclusion

Compare MAE, RMSE, and MAPE in the table above. Lower values indicate better forecasts. Blank LSTM rows mean that the corresponding LSTM result file has not yet been generated. Conclusions about whether TimesFM performs better or worse should be recorded only after all models have been evaluated on compatible chronological test periods. TimesFM's result is a zero-shot pretrained baseline, while the LSTM models learn project-specific patterns during training.